# gem

> Simple utilities for working with Google's Gemini API

This notebook provides a minimal interface to Google's Gemini API. The goal is to make it dead simple to:

1. Generate text with just a prompt
2. Analyze files (PDFs, images, **MP4 videos**) 
3. Process videos (YouTube URLs or **local MP4 files**)

All through a single `gem()` function that just works.

In [ ]:
#| default_exp gem

In [34]:
#| hide
from nbdev.showdoc import *

## Setup

First, make sure you have your Gemini API key set:

In [35]:
#| export
import os, time
from pathlib import Path
from fastcore.all import *
from google import genai
from google.genai import types
from functools import partial
from fastprogress import progress_bar

In [36]:
# export GEMINI_API_KEY='your-api-key'
assert os.environ.get("GEMINI_API_KEY"), "Please set GEMINI_API_KEY environment variable"

## Building blocks

Let's start with the simple helper functions that make everything work.

### Client creation

We need a Gemini client to talk to the API:

In [37]:
#|export
def _client():
    "Get Gemini client context manager"
    return genai.Client()

In [38]:
#|hide
c = _client()
assert c is not None
assert hasattr(c, 'models')

## Video upload

In [39]:
#|export
def upload_file(pth):
    if not Path(pth).exists(): raise ValueError(f"File {pth} does not exist.")
    with _client() as c:
        f = c.files.upload(file=pth)
        time.sleep(2)
        for i in progress_bar(range(30)):
            try:
                f = c.files.get(name=f.name)
                if f.state == 'ACTIVE': return f
                elif f.state == 'FAILED': raise Exception(f'File processing for {pth} failed.')
                time.sleep(10)
            except: pass # because the gemini file thing is jank
        raise Exception(f'Timeout processing {pth}')

In [40]:
myfile = upload_file("_videos/test_video.mp4")
assert myfile.state == 'ACTIVE'

 |----------------------------------------| 0.00% [0/30 00:00<?]

### Converting attachments to Parts

Gemini expects different types of content (files, URLs) to be wrapped in "Parts". This helper handles that conversion:

In [41]:
#| export
def _is_url(s):
    "Check if string is a URL"
    if not isinstance(s, str): return False
    return (s.startswith('http://') or 
            s.startswith('https://') or 
            s.startswith('www.') or 
            'youtube.com' in s or 
            'youtu.be' in s)

def _make_part(o):
    "Convert object to Gemini Part"
    if isinstance(o, types.File):
        return types.Part.from_uri(file_uri=o.uri, mime_type=o.mime_type)
    if isinstance(o, (str, Path)):
        p = Path(o)
        if p.exists():
            if p.suffix.lower() == '.mp4':
                f = upload_file(o)
                return types.Part.from_uri(file_uri=f.uri, mime_type=f.mime_type)
            mime_map = {'.pdf': 'application/pdf', 
                        '.png': 'image/png', 
                        '.jpg': 'image/jpeg', 
                        '.jpeg': 'image/jpeg', 
                        '.gif': 'image/gif'}
            mime = mime_map.get(p.suffix.lower(), 'application/octet-stream')
            return types.Part.from_bytes(mime_type=mime, data=p.read_bytes())
        elif _is_url(o): return types.Part.from_uri(file_uri=o, mime_type='video/*')
        else: raise ValueError(f"Could not parse file or url: {o}")
    return None

In [42]:
_part = _make_part('_videos/test_video.mp4')
_part

 |----------------------------------------| 0.00% [0/30 00:00<?]

Part(
  file_data=FileData(
    file_uri='https://generativelanguage.googleapis.com/v1beta/files/y1fsnubeebrs',
    mime_type='video/mp4'
  )
)

## The main interface

Now we can build our main `gem()` function that handles all use cases:

In [43]:
#| export
def gem(prompt, # Text prompt
        o=None, # Optional file/URL attachment or list of attachments
        model='gemini-2.5-flash',
        thinking=-1,
        search=False):
    "Generate content with Gemini"
    parts = [types.Part.from_text(text=prompt)]
    # Handle single attachment or list of attachments
    attachments = o if isinstance(o, list) else [o] if o else []
    for attachment in attachments:
        if part := _make_part(attachment): parts.insert(0, part)
    
    contents = types.Content(role='user', parts=parts) if attachments else prompt    
    config_dict = {
        'thinking_config': types.ThinkingConfig(thinking_budget=thinking),
        'response_mime_type': 'text/plain'
    }
    # Adjust media_resolution for videos for more tokens
    if any(p.file_data and p.file_data.mime_type.startswith('video') for p in parts):
        config_dict['media_resolution'] = 'MEDIA_RESOLUTION_LOW'
    config_dict['tools'] = []
    if search: config_dict['tools'].append(types.Tool(google_search=types.GoogleSearch()))
    cfg = types.GenerateContentConfig(**config_dict)
    with _client() as client:
        resp = client.models.generate_content(model=model, contents=contents, config=cfg)
    return resp.text

## Examples

One function handles everything:
- Just text? Pass a prompt.
- Have a file? Pass it as the second argument.
- Got a YouTube URL? Same thing.

Let's test it out:

## Text generation

The simplest case - just generate some text:

In [44]:
gem("Write a haiku about Python programming")

'Clean lines, space for thought,\nIndentation guides the way,\nPower in each script.'

## Video analysis

Perfect for creating YouTube chapters or summaries:

In [45]:
prompt = "5 word summary of this video."
gem(prompt, "https://youtu.be/1x3k0V2IITo")

'Late interaction: beyond vector limits.'

### Local MP4 Video Analysis

You can also analyze local MP4 video files:

In [47]:
# Example with local MP4 file (if you have one)
gem("Summarize this video in 3 sentences.", "_videos/test_video.mp4")

 |█---------------------------------------| 3.33% [1/30 00:10<04:53]

'A bald man wearing glasses records a brief test video, speaking directly to the camera. He states he will say the numbers one through six, which he proceeds to do. The man concludes the recording by identifying himself as Hamil Hussein.'

### File analysis

Great for extracting information from PDFs or images:

In [48]:
gem("3 sentence summary of this presentation.", "NewFrontiersInIR.pdf")

'This presentation explores new frontiers in Information Retrieval (IR) by enabling systems to follow complex instructions and reason like Large Language Models. It introduces Promptriever, an instruction-trained retriever that can be prompted using natural language, significantly improving performance on instruction-based search tasks. Additionally, Rank1 utilizes test-time compute to implement reasoning for reranking, demonstrating substantial gains on complex reasoning and negation-based retrieval challenges.'

In [49]:
gem("What's in this image?", "anton.png")

'This image appears to be a YouTube thumbnail or similar promotional graphic, likely for a video about technology, specifically related to "vectors" and "RAG" (Retrieval Augmented Generation).\n\nHere\'s a breakdown of what\'s in the image:\n\n1.  **Background:** A dark, solid blue-black color, providing a strong contrast for the text and glowing graphics.\n\n2.  **Text and Emoji (Top Left to Mid-Center):**\n    *   **"Single Vector?"** in large, white, sans-serif font, accompanied by a question mark.\n    *   A prominent **yellow sad/worried emoji** positioned below "Single Vector?".\n    *   **"YOU\'RE MISSING OUT"** in large, bold, bright yellow, sans-serif font, spread across three lines, reinforcing the message linked to the emoji.\n\n3.  **Person (Bottom Left):**\n    *   A young man with light brown hair and a white t-shirt, smiling and looking directly at the viewer. His head and upper chest are visible.\n\n4.  **Abstract Graphics/Diagrams (Right Side):**\n    *   **Top Right:*

### Change Model

You can also control the model and thinking time:

In [50]:
gem("What is Hamel Husain's current job?", model="gemini-2.5-pro")

"Based on his public profiles, Hamel Husain's current job is **Head of Machine Learning at a stealth startup**.\n\nHe was previously a Principal Machine Learning Engineer at **GitHub**, where he was a key figure in the development of products like GitHub Copilot. He is also well-known in the data science community for his work with fast.ai and for creating popular open-source tools like `nbdev`."

### Grounded Search

As you can see, grounded search is required to get things right sometimes!

In [51]:
gem("What is Hamel Husain's current job?.", search=True)

'Hamel Husain is currently an independent consultant specializing in helping companies build and operationalize AI products, with a focus on Large Language Models (LLMs). He is also a machine learning engineer with over 20 years of experience.\n\nIn addition to his consulting work, Husain co-teaches a popular course titled "AI Evals for Engineers & PMs," which has educated over 3,000 students from numerous companies, including OpenAI, Anthropic, and Google. He is also listed as part of the team at Parlance Labs. Previously, he held the position of Staff Machine Learning Engineer at GitHub.'

### Multiple Attachments

You can analyze multiple files/URLs at once by passing a list:

In [52]:
prompt = "Is this PDF and YouTube video related or are they different talks? Answer with very short yes/no answer."
gem(prompt, ["https://youtu.be/Trps2swgeOg?si=yK7CO0Zk4E1rfp6s", "NewFrontiersInIR.pdf"])

'No.'

In [53]:
gem(prompt, ["https://youtu.be/YB3b-wPbSH8?si=WI0LqflY5SYIsRz9", "NewFrontiersInIR.pdf"])

'Yes.'

In [54]:
gem("What do these slides and this video have in common in terms of content/subject matter if at all? Provide a 1 sentence summary of each.", ["NewFrontiersInIR.pdf", "_videos/test_video.mp4"])

 |----------------------------------------| 0.00% [0/30 00:00<?]

'The video and the slides have **no common subject matter in terms of content**. The video is a personal audio test, while the slides present a technical discussion on information retrieval.\n\nHere\'s a one-sentence summary for each:\n\n*   **Video Summary:** The video shows a man conducting a brief audio test by introducing himself and counting.\n*   **Slides Summary:** The slides present "New Frontiers in IR," detailing research on making information retrieval systems (like Promptriever and Rank1) more capable of instruction following and complex reasoning, similar to large language models.'

## Export -

In [55]:
#| hide
import nbdev; nbdev.nbdev_export()